# Hands-On Pertemuan 14: Advanced Machine Learning using Spark MLlib

## Objectives:
- Understand and implement advanced machine learning tasks using Spark MLlib.
- Build and evaluate models using real-world datasets.
- Explore techniques like feature engineering and hyperparameter tuning.


## Introduction to Spark MLlib
Spark MLlib is a scalable library for machine learning that integrates seamlessly with the Spark ecosystem. It supports a wide range of tasks, including regression, classification, clustering, and collaborative filtering.

In [5]:
# Example: Linear Regression with Spark MLlib
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.feature import VectorAssembler

# Initialize Spark Session
spark = SparkSession.builder.appName('MLlib Example').getOrCreate()

# Load sample data
data = [(1, 5.0, 20.0), (2, 10.0, 25.0), (3, 15.0, 30.0), (4, 20.0, 35.0)]
columns = ['ID', 'Feature', 'Target']
df = spark.createDataFrame(data, columns)

# Prepare data for modeling
assembler = VectorAssembler(inputCols=['Feature'], outputCol='Features')
df_transformed = assembler.transform(df)

# Train a linear regression model
lr = LinearRegression(featuresCol='Features', labelCol='Target')
model = lr.fit(df_transformed)

# Print model coefficients
print(f'Coefficients: {model.coefficients}')
print(f'Intercept: {model.intercept}')


Coefficients: [0.9999999999999992]
Intercept: 15.000000000000009


In [6]:
# Practice: Logistic Regression
from pyspark.ml.classification import LogisticRegression
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

spark = SparkSession.builder.appName('MLlib Example2').getOrCreate()

# Example dataset
data = [(1, [2.0, 3.0], 0), (2, [1.0, 5.0], 1), (3, [2.5, 4.5], 1), (4, [3.0, 6.0], 0)]
columns = ['ID', 'Features', 'Label']
df = spark.createDataFrame(data, columns)

# Instead of using 'Features' directly, we need to access the elements within the array
# Create new columns for 'Features[0]' and 'Features[1]' using Spark functions
df = df.withColumn('Features0', col('Features').getItem(0)) \
       .withColumn('Features1', col('Features').getItem(1))

# Now use VectorAssembler with the new columns
assembler = VectorAssembler(inputCols=['Features0', 'Features1'], outputCol='FeaturesVector')
df = assembler.transform(df)

# Train logistic regression model using the 'FeaturesVector' column
lr = LogisticRegression(featuresCol='FeaturesVector', labelCol='Label')
model = lr.fit(df)

# Display coefficients and summary
print(f'Coefficients: {model.coefficients}')
print(f'Intercept: {model.intercept}')

Coefficients: [-12.262057937838394,4.087352269372807]
Intercept: 11.568912735310269


In [7]:

# Practice: Kmeans Clustering
from pyspark.ml.clustering import KMeans
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col

# Initialize Spark session
spark = SparkSession.builder.appName('KMeans Example').getOrCreate()

# Example dataset
data = [(1, [1.0, 1.0]), (2, [5.0, 5.0]), (3, [10.0, 10.0]), (4, [15.0, 15.0])]
columns = ['ID', 'Features']
df = spark.createDataFrame(data, columns)

# Extract elements from 'Features' array into separate columns
df = df.withColumn('Features0', col('Features').getItem(0)) \
       .withColumn('Features1', col('Features').getItem(1))

# Use VectorAssembler with the new columns
assembler = VectorAssembler(inputCols=['Features0', 'Features1'], outputCol='FeaturesVector')
df_vector = assembler.transform(df)

# Train KMeans clustering model using the 'FeaturesVector' column
kmeans = KMeans(featuresCol='FeaturesVector', k=2)
model = kmeans.fit(df_vector)

# Show cluster centers
centers = model.clusterCenters()
print(f'Cluster Centers: {centers}')

# Optionally, show the dataset with cluster assignments
df_clusters = model.transform(df_vector)
df_clusters.select('ID', 'Features', 'FeaturesVector', 'prediction').show()


Cluster Centers: [array([12.5, 12.5]), array([3., 3.])]
+---+------------+--------------+----------+
| ID|    Features|FeaturesVector|prediction|
+---+------------+--------------+----------+
|  1|  [1.0, 1.0]|     [1.0,1.0]|         1|
|  2|  [5.0, 5.0]|     [5.0,5.0]|         1|
|  3|[10.0, 10.0]|   [10.0,10.0]|         0|
|  4|[15.0, 15.0]|   [15.0,15.0]|         0|
+---+------------+--------------+----------+



## Homework
- Load a real-world dataset into Spark and prepare it for machine learning tasks.
- Build a classification model using Spark MLlib and evaluate its performance.
- Explore hyperparameter tuning using cross-validation.


In [1]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1. Load
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col

# Membuat SparkSession
spark = SparkSession.builder.appName("Data Titanic").getOrCreate()

def muat_data_ke_spark(file_path):
    dataset = spark.read.csv(file_path, header=True, inferSchema=True)
    return dataset

# Memuat data
dataset_spark = muat_data_ke_spark("C:\\Users\\AKBAR\\Downloads\\titanic.csv")


In [3]:
# 2. Klasifikasi
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load dataset (ganti path 'titanic.csv' dengan lokasi file Anda)
data = pd.read_csv("C:\\Users\\AKBAR\\Downloads\\titanic.csv")

# Preprocessing: Menghapus kolom yang tidak relevan atau terlalu kompleks untuk langsung digunakan
columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin']
data = data.drop(columns=columns_to_drop, axis=1)

# Mengatasi nilai kosong (missing values)
data['Age'] = data['Age'].fillna(data['Age'].median())
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])

# Encode kolom kategorikal
categorical_columns = ['Sex', 'Embarked']
data = pd.get_dummies(data, columns=categorical_columns, drop_first=True)

# Pisahkan fitur (X) dan label (y)
X = data.drop('Survived', axis=1)
y = data['Survived']

# Membagi dataset menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Membuat dan melatih model klasifikasi
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluasi model
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.2f}")

# Menampilkan feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(feature_importance)


Accuracy: 0.82
      Feature  Importance
5    Sex_male    0.273316
4        Fare    0.272058
1         Age    0.252745
0      Pclass    0.078616
2       SibSp    0.052192
3       Parch    0.038490
7  Embarked_S    0.023095
6  Embarked_Q    0.009488


In [4]:
# 3. Cross Validation
from pyspark.sql.functions import when
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession

# Inisialisasi SparkSession
import findspark
findspark.init()

# Konfigurasi eksplisit
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("TitanicSurvivalPredictionCV") \
    .config("spark.driver.host", "localhost") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

def proses_kolom_kategorikal(data_spark):
    kolom_kategorikal = ["Sex", "Embarked", "Pclass"]
    
    # StringIndexer untuk mengubah kategori menjadi indeks numerik
    for kolom in kolom_kategorikal:
        indexer = StringIndexer(inputCol=kolom, outputCol=kolom + "_indexed", handleInvalid="keep")
        data_spark = indexer.fit(data_spark).transform(data_spark)
    
    # OneHotEncoder untuk mengubah indeks menjadi vektor biner (one-hot)
    for kolom in kolom_kategorikal:
        encoder = OneHotEncoder(inputCol=kolom + "_indexed", outputCol=kolom + "_encoded", handleInvalid="keep")
        data_spark = encoder.fit(data_spark).transform(data_spark)
    
    return data_spark

def siapkan_dataset_spark(data_spark):
    # Menangani missing values
    data_spark = data_spark.na.fill({
        'Age': data_spark.select('Age').approxQuantile('Age', [0.5], 0.0)[0],
        'Fare': data_spark.select('Fare').approxQuantile('Fare', [0.5], 0.0)[0],
        'Embarked': data_spark.select('Embarked').groupBy('Embarked').count().orderBy('count', ascending=False).first()[0]
    })
    
    return data_spark

def buat_model_klasifikasi_dengan_cross_validation(data_siap):
    # Merancang fitur
    fitur = [
        "Age", "Fare", "SibSp", "Parch", 
        "Pclass_encoded", "Sex_encoded", "Embarked_encoded"
    ]
    
    # Assembler untuk menggabungkan fitur
    assembler = VectorAssembler(inputCols=fitur, outputCol="features", handleInvalid="skip")
    data_final = assembler.transform(data_siap)
    
    # Membagi data menjadi train dan test
    train, test = data_final.randomSplit([0.8, 0.2], seed=42)
    
    # Membuat model Logistic Regression
    lr = LogisticRegression(featuresCol="features", labelCol="Survived")
    
    # Parameter Grid untuk hyperparameter tuning
    paramGrid = (ParamGridBuilder()
        .addGrid(lr.regParam, [0.01, 0.1, 0.5])  # Regularization strength
        .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])  # Elastic Net mixing parameter
        .addGrid(lr.maxIter, [10, 50, 100])  # Maximum iterations
        .build())
    
    # Evaluator
    evaluator = MulticlassClassificationEvaluator(
        labelCol="Survived", 
        predictionCol="prediction", 
        metricName="accuracy"
    )
    
    # Cross Validator
    cv = CrossValidator(
        estimator=lr,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        numFolds=5  # 5-Fold Cross Validation
    )
    
    print("Memulai Cross Validation...")
    # Fit Cross Validator
    cvModel = cv.fit(train)
    
    # Dapatkan model terbaik
    best_model = cvModel.bestModel
    
    # Evaluasi pada test set
    prediksi = best_model.transform(test)
    
    # Hitung metrik evaluasi
    akurasi = evaluator.evaluate(prediksi)
    
    # Evaluasi tambahan
    precision = MulticlassClassificationEvaluator(
        labelCol="Survived", 
        predictionCol="prediction", 
        metricName="weightedPrecision"
    ).evaluate(prediksi)
    
    recall = MulticlassClassificationEvaluator(
        labelCol="Survived", 
        predictionCol="prediction", 
        metricName="weightedRecall"
    ).evaluate(prediksi)
    
    f1_score = MulticlassClassificationEvaluator(
        labelCol="Survived", 
        predictionCol="prediction", 
        metricName="f1"
    ).evaluate(prediksi)
    
    # Tampilkan hasil evaluasi
    print("Hasil Cross Validation:")
    print(f"Akurasi Terbaik: {akurasi:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1_score:.4f}")
    
    # Cetak parameter terbaik
    print("\nParameter Terbaik:")
    best_params = best_model.extractParamMap()
    for param, value in best_params.items():
        print(f"{param.name}: {value}")
    
    # Simpan model terbaik
    model_path = "./titanic_survival_cv_model"
    best_model.save(model_path)
    print(f"\nModel terbaik disimpan di {model_path}")
    
    return akurasi

def main():
    try:
        # Baca dataset Titanic
        dataset_spark = spark.read.csv(
            "C:\\Users\\AKBAR\\Downloads\\titanic.csv",
            header=True, 
            inferSchema=True
        )
        
        # Tampilkan skema awal
        dataset_spark.printSchema()
        
        # Proses kolom kategorikal
        print("Proses pengolahan kolom kategorikal dimulai...")
        dataset_siap = proses_kolom_kategorikal(dataset_spark)
        print("Proses pengolahan kolom kategorikal selesai.")
        
        # Siapkan dataset
        print("Menyiapkan dataset...")
        dataset_siap = siapkan_dataset_spark(dataset_siap)
        print("Dataset selesai disiapkan.")
        
        # Buat dan evaluasi model dengan Cross Validation
        akurasi_model = buat_model_klasifikasi_dengan_cross_validation(dataset_siap)
        print("Akurasi Model Terbaik:", akurasi_model)
    
    except Exception as e:
        print(f"Terjadi kesalahan: {e}")
    finally:
        # Hentikan SparkSession
        spark.stop()

# Jalankan script
if __name__ == "__main__":
    main()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)

Proses pengolahan kolom kategorikal dimulai...
Proses pengolahan kolom kategorikal selesai.
Menyiapkan dataset...
Dataset selesai disiapkan.
Memulai Cross Validation...
Hasil Cross Validation:
Akurasi Terbaik: 0.8069
Precision: 0.8109
Recall: 0.8069
F1 Score: 0.8049

Parameter Terbaik:
aggregationDepth: 2
elasticNetParam: 0.0
family: auto
featuresCol: features
fitIntercept: True
labelCol: Survived
maxBlockSizeInMB: 0.0
maxIter: 10
predictionCol: prediction
probabilityCol: probability
rawPredictionCol: rawPrediction
regParam: 0.01
s